# DP2 Multiband Injection Demo
Load one DP2 patch, inject 100 clusters in g/r/i, detect with MCI, match recoveries, and plot the results. Run the cells from top to bottom.

## 1. Imports
Run in the CANFAR Rubin kernel with INJECT installed or importable from your checkout.

In [ ]:
from pathlib import Path
import sys
repo = Path('/arc/home/snair25/INJECT')
if repo.is_dir():
    sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import make_lupton_rgb
from lsst.daf.butler import Butler
from inject import ClusterConfig, ClusterRetrieval, InjectionConfig, InjectionPipeline
from inject.detection import run_cluster_detection

## 2. Settings
Use your working DP2 patch. Change `DATA_ID` or `BANDS` if needed. The example uses 100 clusters, one iteration, and a King profile.

In [ ]:
REPO = 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml'
COLLECTION = 'dp2'
DATA_ID = {'skymap': 'lsst_cells_v2', 'tract': 7508, 'patch': 0}
BANDS = ['g', 'r', 'i']
N_CLUSTERS = 100
MATCH_RADIUS = 5.0
OUT = Path('plots/dp2_multiband_injection_demo')
OUT.mkdir(parents=True, exist_ok=True)
config = InjectionConfig(
    bands=BANDS, band=BANDS[0], n_clusters=N_CLUSTERS, cutout_size=800,
    edge_buffer=60, seed=42, use_actual_psf=True, add_noise=False,
    pixel_scale=0.2, zero_point=31.4, output_dir=str(OUT),
    cluster_config=ClusterConfig(
        profile_type='king', mag_min=20.5, mag_max=24.5,
        r_half_min=2, r_half_max=6, concentration_min=5, concentration_max=20,
    ),
)

## 3. Load DP2
Load the same patch in each band. INJECT handles DP2 CellCoadd conversion.

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID, dataset_type='deep_coadd')
print({band: image.shape for band, image in pipe.images.items()})

## 4. Inject and detect
`run_batch_multiband()` makes one shared catalog and uses each band's image and Rubin PSF. MCI runs per band; one-to-one matching uses `MATCH_RADIUS` pixels, so a detection cannot recover multiple injected sources.

In [ ]:
def detect(image):
    return run_cluster_detection(
        image, psf_fwhm=3.5, pixel_scale=config.pixel_scale,
        threshold_sigma=3, snr_min=3, mci_max=0.85, r_half_min=1,
        ellipticity_max=0.6, box_size=64, use_multiscale=True,
        use_mci=True, verbose=False,
    )

baseline_detections = {band: detect(pipe.images[band]) for band in BANDS}
results = pipe.run_batch_multiband(
    n_iterations=1, n_per_iter=N_CLUSTERS, detector_fn=detect,
    checkpoint_dir=str(OUT / 'checkpoints'), store_images=True,
    use_psf_cache=False, n_workers=1, verbose=False,
)

retrievals = {}
summary = []
for band in BANDS:
    result = results[band][0]
    retrieval = ClusterRetrieval(result['injection_info'], result['detections'])
    retrieval.match_detections(
        match_radius=MATCH_RADIUS, baseline_detections=baseline_detections[band],
    )
    retrievals[band] = retrieval
    summary.append({'band': band, **retrieval.get_summary_statistics()})
pd.DataFrame(summary)

## 5. Plot
Save original/injected RGB images, per-band recovered/missed positions, and 1D/2D completeness plots.

In [ ]:
PLOTS = ['injection_summary', 'before_after', 'position_map',
         'completeness_1d', 'completeness_2d', 'psf_fwhm_hist']
for band in BANDS:
    pipe.image = pipe.images[band]
    pipe.injected_image = results[band][0]['injected_image']
    pipe.injection_info = retrievals[band]._matched
    pipe.detection_catalog = results[band][0]['detections']
    pipe.retrieval = retrievals[band]
    pipe.make_plots(
        output_dir=str(OUT / band), plots=PLOTS, show=True, save=True, n_stamps=3,
    )

rgb_bands = ('i', 'r', 'g')
original = [pipe.images[b] - np.nanmedian(pipe.images[b]) for b in rgb_bands]
injected = [results[b][0]['injected_image'] - np.nanmedian(pipe.images[b]) for b in rgb_bands]
stretch = max(float(np.nanpercentile(np.abs(injected), 99.5)), 1e-6)
for name, channels in [('original', original), ('injected', injected)]:
    rgb = make_lupton_rgb(*channels, stretch=stretch, Q=8)
    plt.imsave(OUT / f'{name}_rgb.png', rgb, origin='lower')
    plt.figure(figsize=(7, 7)); plt.imshow(rgb, origin='lower'); plt.axis('off'); plt.show()
print('Saved plots to', OUT)

The RGB image is for display; detection and completeness use each filter separately. This array-based injection demo does not update the Rubin variance plane.